# RAG pipeline з Minsearch

## Налаштування RAG

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from google import genai
gemini_client = genai.Client()

In [5]:
documents = load_faq_data()
index = build_index(documents)

assistant = RAGBase(index, gemini_client)

## Тестування

Задамо кілька запитань.

In [6]:
assistant.rag("I just discovered the course. Can I join now?")

'Yes, you can join now. You don\'t need to register to start learning or submitting homework. However, please note that if you want to receive a certificate, you must submit your project while submissions are still being accepted, as certificates are only awarded to those who finish with a "live" cohort and complete the required peer reviews.'

In [8]:
answer = assistant.rag("How do I get a certificate?")
print(answer)

To get a certificate, you must follow these steps and requirements:

*   **Participate in a "live" cohort:** You cannot earn a certificate in self-paced mode. You must be enrolled during the time the course is running live.
*   **Complete the Capstone project:** You must pass the Capstone project to be eligible. 
*   **Complete peer reviews:** You are required to peer-review 3 capstone projects after submitting your own. This is only possible when the course is running and the peer-review list is compiled.
*   **Provide your official name:** To ensure your certificate has your correct name, you must update your "Edit Course Profile" settings. Use the second field to enter your name exactly as it appears on your official identification documents (passport, driver's license, etc.).

**Note:** Homework is not mandatory for receiving a certificate, though it is recommended for your learning and for your rank on the leaderboard.


In [9]:
answer = assistant.rag("Can I still join the course after it started?")
print(answer)

Yes, you can still join the course after it has started. 

However, if your goal is to receive a certificate, you must ensure you submit your Capstone project while submissions are still being accepted. Additionally, keep in mind that certificates are only awarded to those who participate with a "live" cohort, as you are required to peer-review three other capstone projects during that active period.


Ми отримуємо цілком задовільні відповіді. Але...

## Недоліки конвеєра RAG

Створений нами конвеєр добре працює, коли нейронка отримує з пошуку результати, релевантні запиту. Якщо потрібна інформація не буде знайдена, вона або скаже, що не знає відповіді, або видасть нерелевантну відповідь.

Ми використовуємо текстовий пошук, а отже шукаємо прямі входження ключових слів. Якщо користувач неправильно сформулює питання або зробить помилку, він не отимає потрібну йому інформацію.

Спробуємо протестувати випадок з помилкою. Поставимо питання: "Як запустити Ollama локально?"

In [5]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

/workspaces/LLM_Zoomcamp_2026/rag_helper.py:77: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.llm_client.interactions.create(


To run Ollama locally, follow these steps:

1.  **Install Ollama:** Visit [https://ollama.com/download](https://ollama.com/download) and download/install the version for your operating system (macOS, Windows, or Linux). For Linux, you can use the terminal command:
    `curl -fsSL https://ollama.com/install.sh | sh`
2.  **Start the model:** Open a terminal and type:
    `ollama run llama3`
    This will download the LLaMA 3 model and open a chat-like interface.
3.  **Verify the server:** You can test that the local server is running by executing:
    `curl http://localhost:11434`
4.  **Use it in Python:** Install the Python client with `pip install ollama` and use the following code:
    ```python
    import ollama

    response = ollama.chat(
        model='llama3',
        messages=[{"role": "user", "content": your_prompt}]
    )

    print(response['message']['content'])
    ```


Відповідь отримано. А тепер зробимо помилку і пропустимо букву "l" в слові Ollama.

In [6]:
answer = assistant.rag("How do I run Olama locally?")
print(answer)

/workspaces/LLM_Zoomcamp_2026/rag_helper.py:77: UserWarning: Interactions usage is experimental and may change in future versions.
  interaction = self.llm_client.interactions.create(


I don't know.


Отримано відповідь: "я не знаю".

Щоб виправити таку ситуацію, нам знадобляться агенти.